# M5 – Daten behalten

*Begleit-Notebook zu Modul **K5 „Daten behalten“** im Kompaktkurs „Vom Web zur Desktop-App mit Tauri“*

Ihre Lernkarten-App funktioniert, aber sie hat **kein Gedächtnis**: Nach jedem Neustart sind Ihre Karten weg. In diesem Modul schlagen Sie die Brücke zum zweiten Teil Ihrer Anwendung, dem **Programmkern** in Rust. Er schreibt die Karten in eine Datei, und sie überleben den Neustart.

Das ist der Punkt, an dem sich Ihre Anwendung von einer Webseite unterscheidet.

## 🎯 Das können Sie am Ende

- erklären, warum eine Webseite keine Dateien schreiben darf, eine installierte Anwendung aber schon
- die zwei Teile der Anwendung benennen: **Fenster** und **Programmkern**
- die beiden vorbereiteten Funktionen in `main.rs` in eigenen Worten beschreiben
- mit `invoke`, `await` und `async` den Programmkern aufrufen, als **Rezept**
- `speichern()` und `laden()` schreiben und an den richtigen Stellen aufrufen
- den Merksatz anwenden: **Wo `await` steht, muss `async` davorstehen.**

## 🧭 Inhalt

0. Vorbereitung
1. Das Problem: Nach dem Neustart ist alles weg
2. Die zwei Teile einer Desktop-Anwendung
3. Der vorbereitete Programmkern
4. Der erste Aufruf: `speichern()`
5. Laden beim Start
6. Speichern an den richtigen Stellen
7. Absichtlicher Fehler: das fehlende `await`
8. Fehlerjagd
9. Wo liegt die Datei? Und eine Werkstatt für Schnellere
10. Diagnosetabelle, Checkliste und Ausblick auf K6

## So funktioniert dieses Notebook

Wie in M4 öffnet jede Zelle, die mit **`// app.js`** beginnt, ein kleines Anwendungsfenster mit Konsole. Neu sind zwei Dinge:

| Neu | Wozu |
|---|---|
| Knopf **⟳ Neu starten** oben im Fenster | schließt die App und startet sie neu, so wie Schließen und Öffnen der echten Anwendung |
| Bereich **📄 lernkarten.json** unter der Konsole | zeigt, was gerade in der Datei steht, und meldet jeden Zugriff des Programmkerns |
| Knopf **🗑 Datei löschen** | setzt die Datei zurück, als wäre die App frisch installiert |

> **📌 Ehrlicher Hinweis: Hier spielt die Werkzeugkiste den Programmkern.**
> Im Notebook gibt es kein Rust. Die Vorbereitungszelle bildet die beiden Funktionen aus `main.rs` nach und legt `lernkarten.json` im Speicher Ihres Browsers ab. Für Ihr JavaScript sieht das genauso aus wie in der echten App: Sie rufen `window.__TAURI__.core.invoke(...)` auf und bekommen dieselben Antworten und Fehlermeldungen.
> Die Datei bleibt erhalten, auch wenn Sie eine Zelle erneut ausführen oder das Notebook schließen. Alle Fenster in diesem Notebook teilen sich **eine** Datei.

Die Zeichen kennen Sie: 🤔 **Vorhersage** · ✏️ **Ihre Aufgabe** · 🐞 **Absichtlicher Fehler** · 🖱️ **Klicken Sie** · 💡 **Lösung**.

---
## 0 Vorbereitung

Führen Sie die eingeklappte Zelle **als Erstes** aus. Sie enthält die Werkzeugkiste aus M4 mit dem nachgebildeten Programmkern und Ihre `index.html` und `style.css` aus K1 und K2.

Wenn darunter **🧰 Werkzeugkiste bereit** steht, kann es losgehen.

> Wer seine eigene Oberfläche aus M4 verwenden möchte, kopiert die `index.html`- und `style.css`-Zellen von dort hierher und führt sie nach der Vorbereitung aus.

In [ ]:
// Vorbereitete Werkzeugkiste für M5: nur ausführen, nicht lesen müssen.
// Sie verwendet Schreibweisen, die im Kurs bewusst NICHT vorkommen.
// Was sie tut: Jede Zelle, die mit "// app.js" beginnt, wird nicht hier im
// Notebook ausgeführt, sondern in einem kleinen Anwendungsfenster mit
// index.html und style.css – samt eigener Konsole darunter.
// Neu in M5: Sie spielt auch den Programmkern (main.rs). window.__TAURI__.core.invoke
// kennt "karten_laden" und "karten_speichern" und hält lernkarten.json im Speicher
// des Browsers fest, damit die Karten einen Neustart überleben.
(function () {
  var STANDARD_HTML = "<!-- index.html -->\n<!DOCTYPE html>\n<html lang=\"de\">\n  <head>\n    <meta charset=\"UTF-8\" />\n    <title>Lernkarten</title>\n    <link rel=\"stylesheet\" href=\"style.css\" />\n  </head>\n  <body>\n    <h1>Lernkarten von Anna</h1>\n\n    <div id=\"eingabe\">\n      <h2>Neue Karte</h2>\n      <input id=\"frage-feld\" type=\"text\" placeholder=\"Frage\" />\n      <textarea id=\"antwort-feld\" placeholder=\"Antwort\"></textarea>\n      <button id=\"knopf-neu\">Karte anlegen</button>\n    </div>\n\n    <div id=\"anzeige\">\n      <p id=\"karte-frage\">Noch keine Karte</p>\n      <p id=\"karte-antwort\"></p>\n      <div id=\"knopf-leiste\">\n        <button id=\"knopf-umdrehen\" class=\"knopf-anzeige\">Umdrehen</button>\n        <button id=\"knopf-weiter\" class=\"knopf-anzeige\">Nächste</button>\n      </div>\n    </div>\n\n    <h2>Alle Karten</h2>\n    <ul id=\"karten-liste\"></ul>\n\n    <script src=\"app.js\"></script>\n  </body>\n</html>";
  var STANDARD_CSS = "/* style.css */\nbody {\n  background-color: #f4f4f4;\n  color: #222222;\n  font-family: sans-serif;\n}\n\nbutton {\n  background-color: #2a4d69;\n  color: #ffffff;\n  border: none;\n  border-radius: 6px;\n  padding: 8px 16px;\n  cursor: pointer;\n}\n\nbutton:hover {\n  background-color: #3d6b91;\n}\n\n.knopf-anzeige {\n  background-color: #6b8f71;\n}\n\n#eingabe {\n  display: flex;\n  flex-direction: column;\n  gap: 12px;\n  max-width: 400px;\n}\n\ninput {\n  border: 1px solid #cccccc;\n  border-radius: 6px;\n  padding: 8px;\n  font-size: 16px;\n  font-family: sans-serif;\n}\n\ntextarea {\n  border: 1px solid #cccccc;\n  border-radius: 6px;\n  padding: 8px;\n  font-size: 16px;\n  font-family: sans-serif;\n}\n\n#anzeige {\n  display: flex;\n  flex-direction: column;\n  gap: 12px;\n  background-color: #ffffff;\n  border: 2px solid #cccccc;\n  border-radius: 12px;\n  padding: 20px;\n  margin-top: 24px;\n  max-width: 400px;\n}\n\n#karte-frage {\n  font-size: 20px;\n  font-weight: 700;\n}\n\n#knopf-leiste {\n  display: flex;\n  gap: 12px;\n}";

  // ---------- Ausgabe im Notebook selbst ----------
  function zeige(wert, tiefe) {
    if (wert === undefined) { return "undefined"; }
    if (wert === null) { return "null"; }
    if (typeof wert === "string") { return tiefe === 0 ? wert : '"' + wert + '"'; }
    if (typeof wert === "function") { return "ƒ " + (wert.name || "anonym") + "()"; }
    if (Array.isArray(wert)) {
      return "[" + wert.map(function (w) { return zeige(w, tiefe + 1); }).join(", ") + "]";
    }
    if (typeof wert === "object") {
      return "{ " + Object.keys(wert).map(function (f) { return f + ": " + zeige(wert[f], tiefe + 1); }).join(", ") + " }";
    }
    return String(wert);
  }
  function ausgabe(kanal, werte) {
    var text = Array.prototype.map.call(werte, function (w) { return zeige(w, 0); }).join(" ");
    postMessage({ type: "stream", bundle: { name: kanal, text: text + "\n" } });
  }
  console.log = function () { ausgabe("stdout", arguments); };
  console.info = console.log;
  console.error = function () { ausgabe("stderr", arguments); };
  console.warn = console.error;

  // ---------- Die drei Dateien der App ----------
  var dateien = { html: STANDARD_HTML, css: STANDARD_CSS };
  var laeufe = 0;

  function ohneKennung(text) {
    return text.replace(/^\s*(<!--\s*index\.html\s*-->|\/\*\s*style\.css\s*\*\/)[^\n]*\n?/, "");
  }
  function koerper(html) {
    var treffer = /<body[^>]*>([\s\S]*)<\/body>/i.exec(html);
    var inhalt = treffer ? treffer[1] : html;
    return inhalt.replace(/<script[^>]*src=["']?app\.js["']?[^>]*>\s*<\/script>/i, "");
  }
  function titel(html) {
    var treffer = /<title>([\s\S]*?)<\/title>/i.exec(html);
    return treffer ? treffer[1].trim() : "Lernkarten";
  }
  function attribut(text) {
    return text.replace(/&/g, "&amp;").replace(/"/g, "&quot;");
  }
  function htmlText(text) {
    return text.replace(/&/g, "&amp;").replace(/</g, "&lt;").replace(/>/g, "&gt;");
  }

  // Läuft IM Fenster: leitet console.log und Fehler in die Konsole unter dem Fenster um.
  function fensterSkript(versatz) {
    return "(" + function (VERSATZ) {
      var rahmen = window.frameElement ? window.frameElement.closest(".lk-fenster") : null;
      var zeilen = rahmen ? rahmen.querySelector(".lk-konsole") : null;
      var original = window.console.log.bind(window.console);
      if (zeilen) { zeilen.innerHTML = ""; }
      function zeige(w, tiefe) {
        if (w === undefined) { return "undefined"; }
        if (w === null) { return "null"; }
        if (typeof w === "string") { return tiefe === 0 ? w : '"' + w + '"'; }
        if (typeof w === "function") { return "ƒ " + (w.name || "anonym") + "()"; }
        if (w instanceof Promise) { return "Promise { <pending> }"; }
        if (w instanceof Element) {
          var t = "<" + w.tagName.toLowerCase();
          if (w.id) { t += ' id="' + w.id + '"'; }
          if (w.className) { t += ' class="' + w.className + '"'; }
          return t + ">";
        }
        if (Array.isArray(w)) {
          if (w.length === 0) { return "[]"; }
          var teile = w.map(function (x) { return zeige(x, tiefe + 1); });
          var mitObjekten = w.some(function (x) { return x !== null && typeof x === "object"; });
          if (mitObjekten) {
            var ein = "  ".repeat(tiefe + 1);
            return "[\n" + ein + teile.join(",\n" + ein) + "\n" + "  ".repeat(tiefe) + "]";
          }
          return "[" + teile.join(", ") + "]";
        }
        if (typeof w === "object") {
          var felder = Object.keys(w);
          if (felder.length === 0) { return "{}"; }
          return "{ " + felder.map(function (f) { return f + ": " + zeige(w[f], tiefe + 1); }).join(", ") + " }";
        }
        return String(w);
      }
      function zeile(text, fehler) {
        if (!zeilen) { return; }
        var leer = zeilen.querySelector(".lk-leer");
        if (leer) { leer.remove(); }
        var div = zeilen.ownerDocument.createElement("div");
        div.textContent = text;
        div.style.whiteSpace = "pre-wrap";
        div.style.padding = "2px 0";
        div.style.borderBottom = "1px solid rgba(128,128,128,.2)";
        if (fehler) {
          div.style.color = "#b3261e";
          div.style.background = "#fdecea";
          div.style.padding = "2px 4px";
        }
        zeilen.appendChild(div);
      }
      window.console.log = function () {
        original.apply(null, arguments);
        zeile(Array.prototype.map.call(arguments, function (w) { return zeige(w, 0); }).join(" "), false);
      };
      window.console.info = window.console.log;
      window.console.error = function () {
        zeile(Array.prototype.map.call(arguments, function (w) { return zeige(w, 0); }).join(" "), true);
      };
      window.console.warn = window.console.error;
      window.addEventListener("error", function (e) {
        var nummer = e.lineno - VERSATZ;
        var ort = nummer > 0 ? "    (app.js, Zeile " + nummer + ")" : "";
        zeile("❌ " + e.message + ort, true);
      });
      window.addEventListener("unhandledrejection", function (e) {
        var grund = e.reason;
        var text = typeof grund === "string" ? grund : (grund && grund.name ? grund.name + ": " + grund.message : String(grund));
        var ort = "";
        var stelle = grund && grund.stack ? /about:srcdoc:(\d+):\d+/.exec(grund.stack) : null;
        if (stelle && Number(stelle[1]) - VERSATZ > 0) { ort = "    (app.js, Zeile " + (Number(stelle[1]) - VERSATZ) + ")"; }
        zeile("❌ Uncaught (in promise) " + text + ort, true);
      });

      // ---------- Der nachgebildete Programmkern ----------
      var SCHLUESSEL = "tauri-kompaktkurs/lernkarten.json";
      var dateiFeld = rahmen ? rahmen.querySelector(".lk-datei") : null;
      var dateiInfo = rahmen ? rahmen.querySelector(".lk-datei-info") : null;
      function dateiLesen() {
        try { return window.localStorage.getItem(SCHLUESSEL); } catch (f) { return null; }
      }
      function dateiZeigen(aktion) {
        var text = dateiLesen();
        if (dateiFeld) { dateiFeld.textContent = text === null ? "(Die Datei gibt es noch nicht.)" : text; }
        if (dateiInfo) {
          var jetzt = new Date();
          var zeit = ("0" + jetzt.getHours()).slice(-2) + ":" + ("0" + jetzt.getMinutes()).slice(-2) + ":" + ("0" + jetzt.getSeconds()).slice(-2);
          dateiInfo.textContent = aktion ? "· " + aktion + " (" + zeit + ")" : "";
        }
      }
      window.__TAURI__ = { core: { invoke: function (name, argumente) {
        return new Promise(function (erfuellt, abgelehnt) {
          setTimeout(function () {
            if (name === "karten_laden") {
              var text = dateiLesen();
              var liste = text === null ? [] : JSON.parse(text);
              dateiZeigen("karten_laden: " + liste.length + " Karten gelesen");
              erfuellt(liste);
            } else if (name === "karten_speichern") {
              if (!argumente || !("karten" in argumente)) {
                abgelehnt("invalid args `karten` for command `karten_speichern`: command karten_speichern missing required key karten");
                return;
              }
              try { window.localStorage.setItem(SCHLUESSEL, JSON.stringify(argumente.karten, null, 2)); } catch (f) { }
              dateiZeigen("karten_speichern: " + (argumente.karten ? argumente.karten.length : 0) + " Karten geschrieben");
              erfuellt(null);
            } else {
              abgelehnt("Command " + name + " not found");
            }
          }, 300);
        });
      } } };
      dateiZeigen("");
      if (rahmen) {
        var neustart = rahmen.querySelector(".lk-neustart");
        var loeschen = rahmen.querySelector(".lk-datei-loeschen");
        var iframe = window.frameElement;
        if (neustart) { neustart.onclick = function () { iframe.srcdoc = iframe.srcdoc; }; }
        if (loeschen) {
          loeschen.onclick = function () {
            try { window.localStorage.removeItem(SCHLUESSEL); } catch (f) { }
            dateiZeigen("Datei gelöscht");
          };
        }
      }
      function hoehe() {
        if (!window.frameElement || !document.body) { return; }
        var unten = document.body.getBoundingClientRect().bottom + window.scrollY;
        var rand = parseFloat(getComputedStyle(document.body).marginBottom) || 0;
        var neu = Math.ceil(unten + rand) + "px";
        if (window.frameElement.style.height !== neu) { window.frameElement.style.height = neu; }
      }
      window.addEventListener("load", hoehe);
      document.addEventListener("DOMContentLoaded", function () {
        new ResizeObserver(hoehe).observe(document.body);
      });
    }.toString() + ")(" + versatz + ");";
  }

  function fenster(appJs, hinweis) {
    laeufe = laeufe + 1;
    var html = ohneKennung(dateien.html);
    var css = ohneKennung(dateien.css);
    var oben = "<!DOCTYPE html>\n<html lang=\"de\">\n<head>\n<meta charset=\"UTF-8\">\n" +
      "<style>\n" + css + "\n</style>\n<script>__SHIM__<\/script>\n</head>\n<body>\n" +
      koerper(html) + "\n<script>\n";
    var versatz = oben.split("\n").length - 1 + fensterSkript(0).split("\n").length - 1;
    var dokument = oben.replace("__SHIM__", fensterSkript(versatz)) + appJs + "\n<\/script>\n</body>\n</html>";
    var leerText = hinweis || "(noch keine Ausgabe)";
    var aussen =
      '<div class="lk-fenster" style="max-width:760px;border:1px solid #9e9e9e;border-radius:10px;overflow:hidden;' +
      'box-shadow:0 3px 12px rgba(0,0,0,.18);margin:6px 0;font-family:system-ui,sans-serif;">' +
      '<div style="background:#303030;color:#f0f0f0;padding:6px 12px;font-size:13px;display:flex;justify-content:space-between;align-items:center;">' +
      '<span>🗂️ ' + htmlText(titel(dateien.html)) + '</span>' +
      '<button class="lk-neustart" title="App schließen und neu starten" style="background:#4a4a4a;color:#f0f0f0;border:1px solid #6a6a6a;border-radius:5px;padding:2px 10px;font-size:12px;cursor:pointer;">⟳ Neu starten</button></div>' +
      '<iframe title="Lernkarten-Fenster" srcdoc="' + attribut(dokument) + '" ' +
      'style="display:block;width:100%;height:420px;border:0;background:#ffffff;"></iframe>' +
      '<div style="border-top:2px solid #9e9e9e;background:#f6f6f6;color:#1f1f1f;font-family:ui-monospace,Menlo,Consolas,monospace;font-size:12.5px;">' +
      '<div style="padding:4px 10px;background:#e4e4e4;color:#444;font-family:system-ui,sans-serif;font-size:12px;">Konsole</div>' +
      '<div class="lk-konsole" style="padding:4px 10px 6px 10px;min-height:20px;">' +
      '<div class="lk-leer" style="color:#888;">' + htmlText(leerText) + '</div></div></div>' +
      '<div style="border-top:2px solid #9e9e9e;background:#fffdf2;color:#1f1f1f;font-size:12px;">' +
      '<div style="padding:4px 10px;background:#eee6c8;color:#444;display:flex;justify-content:space-between;align-items:center;gap:8px;">' +
      '<span>📄 lernkarten.json <span class="lk-datei-info" style="opacity:.75"></span></span>' +
      '<button class="lk-datei-loeschen" title="Datei löschen, als wäre die App frisch installiert" style="background:#fff;color:#444;border:1px solid #bbb;border-radius:5px;padding:1px 8px;font-size:12px;cursor:pointer;">🗑 Datei löschen</button></div>' +
      '<pre class="lk-datei" style="margin:0;padding:6px 10px;max-height:200px;overflow:auto;white-space:pre-wrap;font-family:ui-monospace,Menlo,Consolas,monospace;font-size:12px;background:transparent;color:#1f1f1f;"></pre>' +
      '</div></div>';
    postMessage({
      type: "display_data",
      bundle: { data: { "text/html": aussen, "text/plain": "[Lernkarten-Fenster]" }, metadata: {}, transient: {} }
    });
  }

  if (!self.__lkEvalOriginal) { self.__lkEvalOriginal = self.eval; }
  self.eval = function (code) {
    var anfang = code.replace(/^\s+/, "");
    if (anfang.indexOf("// app.js") === 0) {
      fenster(code);
      return undefined;
    }
    if (anfang.indexOf("<!-- index.html") === 0) {
      dateien.html = code;
      console.log("✅ index.html übernommen. So sieht die Oberfläche ohne app.js aus:");
      fenster("", "(leer: es gibt noch kein app.js)");
      return undefined;
    }
    if (anfang.indexOf("/* style.css") === 0) {
      dateien.css = code;
      console.log("✅ style.css übernommen. So sieht die Oberfläche ohne app.js aus:");
      fenster("", "(leer: es gibt noch kein app.js)");
      return undefined;
    }
    return self.__lkEvalOriginal(code);
  };

  console.log("🧰 Werkzeugkiste bereit. Zellen, die mit // app.js beginnen, öffnen ein Fenster mit Programmkern.");
})();

---
## 1 Das Problem: Nach dem Neustart ist alles weg

Das ist Ihr `app.js` vom Ende von K4. 🖱️ Legen Sie eine eigene Karte an. Klicken Sie dann oben im Fenster auf **⟳ Neu starten**.

In [ ]:
// app.js
const karten = [
  { frage: "Was ist HTML?", antwort: "Die Struktur einer Seite" },
  { frage: "Was ist CSS?", antwort: "Das Aussehen einer Seite" },
  { frage: "Was ist eine id?", antwort: "Der Name eines Elements" }
];

let aktuelleKarte = 0;

function karteAnlegen(frage, antwort) {
  const neueKarte = { frage: frage, antwort: antwort };
  karten.push(neueKarte);
  return neueKarte;
}

function listeAnzeigen() {
  const liste = document.querySelector("#karten-liste");
  liste.innerHTML = "";

  for (const karte of karten) {
    const eintrag = document.createElement("li");
    eintrag.textContent = karte.frage;
    liste.appendChild(eintrag);
  }
}

// Knopf „Karte anlegen“
const knopfNeu = document.querySelector("#knopf-neu");

knopfNeu.addEventListener("click", function () {
  const frage = document.querySelector("#frage-feld").value;
  const antwort = document.querySelector("#antwort-feld").value;

  if (frage === "" || antwort === "") {
    console.log("Bitte Frage und Antwort ausfüllen");
    return;
  }

  karteAnlegen(frage, antwort);
  listeAnzeigen();

  document.querySelector("#frage-feld").value = "";
  document.querySelector("#antwort-feld").value = "";
});

// Knopf „Umdrehen“
const knopfUmdrehen = document.querySelector("#knopf-umdrehen");

knopfUmdrehen.addEventListener("click", function () {
  const karte = karten[aktuelleKarte];
  if (!karte) {
    console.log("Noch keine Karte da");
    return;
  }

  const frageElement = document.querySelector("#karte-frage");
  const antwortElement = document.querySelector("#karte-antwort");

  if (frageElement.textContent === karte.frage) {
    antwortElement.textContent = karte.antwort;
  } else {
    frageElement.textContent = karte.frage;
  }
});

// Knopf „Nächste“
const knopfWeiter = document.querySelector("#knopf-weiter");

knopfWeiter.addEventListener("click", function () {
  if (karten.length === 0) {
    console.log("Noch keine Karte da");
    return;
  }

  aktuelleKarte = (aktuelleKarte + 1) % karten.length;

  document.querySelector("#karte-frage").textContent = karten[aktuelleKarte].frage;
  document.querySelector("#karte-antwort").textContent = "";
});

// Die Anzeige zu Beginn
listeAnzeigen();

Ihre Karte ist weg. Beim Start stehen wieder nur die drei Karten, die im Programmtext stehen. Alles, was beim Benutzen dazukam, lebte nur im Arbeitsspeicher, und der wird beim Schließen geräumt.

### 🤔 Warum kann eine Webseite das nicht?

Überlegen Sie, bevor Sie aufklappen: Warum darf eine Webseite keine Dateien auf Ihren Rechner schreiben?

<details>
<summary><b>💡 Antwort</b> (zum Aufklappen anklicken)</summary>

Eine Webseite kommt von einem **fremden Server**. Sie wird beim Surfen einfach geladen, ohne dass Sie gefragt werden. Dürfte sie in Ihre Dateien schreiben, könnte jede Seite, die Sie besuchen, das auch.

Eine installierte Anwendung haben Sie **bewusst installiert**. Deshalb darf sie das.

> **Eine Webseite ist ein Gast. Eine installierte Anwendung ist eingezogen und hat einen Schlüssel für die Schränke.**
> Der Unterschied ist nicht die Technik, sondern die Erlaubnis.

</details>

---
## 2 Die zwei Teile einer Desktop-Anwendung

```mermaid
flowchart LR
  A["Fenster<br/>index.html + style.css + app.js"] -->|invoke| B["Programmkern<br/>main.rs"]
  B --> C[(lernkarten.json)]
  B -->|Antwort| A
```

| Teil | Was man sieht | Was er darf |
|---|---|---|
| **Fenster** | alles: HTML, CSS, JavaScript | anzeigen, rechnen, auf Klicks reagieren |
| **Programmkern** | nichts | auf die Festplatte zugreifen |

Bis jetzt lief alles im Fenster. Ab jetzt **ruft das Fenster den Programmkern** und bittet ihn, etwas zu tun. Erinnern Sie sich an die zwei Ausgabeorte aus K3? Das **Terminal** zeigt Meldungen des Programmkerns, die **Konsole** zeigt Meldungen aus dem Fenster.

### 🤔 Wer macht was?

| Aufgabe | Fenster oder Programmkern? |
|---|---|
| den Knopf „Karte anlegen“ anzeigen | |
| auf den Klick reagieren | |
| die Liste `karten` im Arbeitsspeicher ergänzen | |
| die Karten in `lernkarten.json` schreiben | |
| die Datei beim Start lesen | |
| die gelesenen Karten in der Liste anzeigen | |

<details>
<summary><b>💡 Auflösung</b> (zum Aufklappen anklicken)</summary>

| Aufgabe | Wer |
|---|---|
| Knopf anzeigen | Fenster |
| auf den Klick reagieren | Fenster |
| `karten` ergänzen | Fenster |
| in `lernkarten.json` schreiben | **Programmkern** |
| die Datei lesen | **Programmkern** |
| gelesene Karten anzeigen | Fenster |

Nur die beiden Zugriffe auf die Datei gehören dem Programmkern. Alles andere bleibt, wo es war.

</details>

---
## 3 Der vorbereitete Programmkern

Die Datei `src-tauri/src/main.rs` ist vollständig vorbereitet. **Rust wird nicht unterrichtet.** Sie lesen sie nur und beschreiben in normalem Deutsch, was passiert.

> Ihre Kursleitung hat die echte Datei. Sie kann im Detail etwas anders aussehen, die beiden Funktionen und ihre Namen sind dieselben.

```rust
use std::fs;
use tauri::Manager;

#[tauri::command]
fn karten_speichern(app: tauri::AppHandle, karten: serde_json::Value) {
    let ordner = app.path().app_data_dir().unwrap();
    fs::create_dir_all(&ordner).unwrap();
    let text = serde_json::to_string_pretty(&karten).unwrap();
    fs::write(ordner.join("lernkarten.json"), text).unwrap();
}

#[tauri::command]
fn karten_laden(app: tauri::AppHandle) -> serde_json::Value {
    let datei = app.path().app_data_dir().unwrap().join("lernkarten.json");
    match fs::read_to_string(datei) {
        Ok(text) => serde_json::from_str(&text).unwrap(),
        Err(_) => serde_json::json!([]),
    }
}

fn main() {
    tauri::Builder::default()
        .invoke_handler(tauri::generate_handler![karten_speichern, karten_laden])
        .run(tauri::generate_context!())
        .unwrap();
}
```

In normalem Deutsch:

| Stelle | Was sie bedeutet |
|---|---|
| `#[tauri::command]` | „Diese Funktion darf vom Fenster aus aufgerufen werden.“ |
| `fn karten_speichern(…, karten: …)` | eine Funktion mit dem Namen `karten_speichern`, sie bekommt `karten` mit |
| `app_data_dir()` | der **Anwendungsdatenordner**, ein fester Ordner nur für diese App |
| `to_string_pretty(&karten)` | die Liste in lesbaren Text verwandeln |
| `fs::write(… "lernkarten.json" …)` | den Text in die Datei schreiben |
| `fn karten_laden(…) -> …` | eine Funktion, die etwas **zurückgibt**, wie `return` in JavaScript |
| `Ok(text) => …` | Datei gefunden: den Text wieder in eine Liste verwandeln |
| `Err(_) => json!([])` | keine Datei da, zum Beispiel beim ersten Start: eine **leere Liste** zurückgeben |
| `generate_handler![…]` | die Liste der Funktionen, die das Fenster aufrufen darf |

| Funktion | Was sie tut |
|---|---|
| `karten_speichern(karten)` | schreibt die Liste als JSON in `lernkarten.json` im Anwendungsdatenordner |
| `karten_laden()` | liest die Datei und gibt die Liste zurück; fehlt die Datei, gibt sie eine leere Liste zurück |

> **Warum nicht das Datei-Plugin von Tauri?** Die Tauri-Plugins brauchen Node und einen Bundler. Der Kurs kommt ohne Node aus. Zwei vorbereitete Funktionen sind der einfachere Weg.

✏️ **Ihre Aufgabe:** Beschreiben Sie jede der beiden Funktionen in **einem** Satz, ohne die Tabelle anzusehen.

---
## 4 Der erste Aufruf: `speichern()`

Diese Funktion ruft den Programmkern und bittet ihn, die Karten zu speichern. Ganz unten wird sie einmal zum Ausprobieren aufgerufen.

🖱️ Führen Sie die Zelle aus und schauen Sie auf die Konsole und auf **📄 lernkarten.json**.

In [ ]:
// app.js
const karten = [
  { frage: "Was ist HTML?", antwort: "Die Struktur einer Seite" },
  { frage: "Was ist CSS?", antwort: "Das Aussehen einer Seite" },
  { frage: "Was ist eine id?", antwort: "Der Name eines Elements" }
];

let aktuelleKarte = 0;

function karteAnlegen(frage, antwort) {
  const neueKarte = { frage: frage, antwort: antwort };
  karten.push(neueKarte);
  return neueKarte;
}

function listeAnzeigen() {
  const liste = document.querySelector("#karten-liste");
  liste.innerHTML = "";

  for (const karte of karten) {
    const eintrag = document.createElement("li");
    eintrag.textContent = karte.frage;
    liste.appendChild(eintrag);
  }
}

// Knopf „Karte anlegen“
const knopfNeu = document.querySelector("#knopf-neu");

knopfNeu.addEventListener("click", function () {
  const frage = document.querySelector("#frage-feld").value;
  const antwort = document.querySelector("#antwort-feld").value;

  if (frage === "" || antwort === "") {
    console.log("Bitte Frage und Antwort ausfüllen");
    return;
  }

  karteAnlegen(frage, antwort);
  listeAnzeigen();

  document.querySelector("#frage-feld").value = "";
  document.querySelector("#antwort-feld").value = "";
});

// Knopf „Umdrehen“
const knopfUmdrehen = document.querySelector("#knopf-umdrehen");

knopfUmdrehen.addEventListener("click", function () {
  const karte = karten[aktuelleKarte];
  if (!karte) {
    console.log("Noch keine Karte da");
    return;
  }

  const frageElement = document.querySelector("#karte-frage");
  const antwortElement = document.querySelector("#karte-antwort");

  if (frageElement.textContent === karte.frage) {
    antwortElement.textContent = karte.antwort;
  } else {
    frageElement.textContent = karte.frage;
  }
});

// Knopf „Nächste“
const knopfWeiter = document.querySelector("#knopf-weiter");

knopfWeiter.addEventListener("click", function () {
  if (karten.length === 0) {
    console.log("Noch keine Karte da");
    return;
  }

  aktuelleKarte = (aktuelleKarte + 1) % karten.length;

  document.querySelector("#karte-frage").textContent = karten[aktuelleKarte].frage;
  document.querySelector("#karte-antwort").textContent = "";
});

// ★ NEU
async function speichern() {
  await window.__TAURI__.core.invoke("karten_speichern", { karten: karten });
  console.log("gespeichert");
}

// Die Anzeige zu Beginn
listeAnzeigen();

// ★ NEU: einmal ausprobieren
speichern();

In der Datei stehen jetzt Ihre drei Karten. Fällt Ihnen etwas auf? Der Text sieht fast genauso aus wie die Liste `karten` in Ihrem `app.js`. Diese Schreibweise heißt **JSON**: Listen und Objekte als Text, so dass sie in einer Datei Platz haben.

### Drei Wörter als Rezept

Diese drei Wörter lernen Sie als **Rezept**, nicht als Konzept:

| Wort | Rezeptregel |
|---|---|
| `invoke` | „Ruf den Programmkern.“ |
| `await` | „Warte, bis er fertig ist.“ |
| `async` | „Diese Funktion darf warten.“ |

> ## Merksatz: Wo `await` steht, muss `async` davorstehen.

Der Aufruf hat drei Teile:

| Teil | Bedeutung |
|---|---|
| `window.__TAURI__.core.invoke` | die Tür zum Programmkern |
| `"karten_speichern"` | der Name der Funktion dort drüben. Er muss **exakt** zum Namen in `main.rs` passen. |
| `{ karten: karten }` | die Daten, die mitgegeben werden |

> **Die geschweiften Klammern sind hier ein eigenes Ding.** `{ karten: karten }` ist die Liste der Angaben für den Aufruf. **Links** steht der Name, den die Rust-Funktion erwartet (`karten: …` in `main.rs`), **rechts** der Wert, der hingeschickt wird: Ihre Liste `karten`.

### 🐞 Absichtlicher Fehler: `await` ohne `async`

🤔 Hier fehlt ein Wort. Was meldet die Konsole?

In [ ]:
// app.js
const karten = [
  { frage: "Was ist HTML?", antwort: "Die Struktur einer Seite" },
  { frage: "Was ist CSS?", antwort: "Das Aussehen einer Seite" },
  { frage: "Was ist eine id?", antwort: "Der Name eines Elements" }
];

let aktuelleKarte = 0;

function karteAnlegen(frage, antwort) {
  const neueKarte = { frage: frage, antwort: antwort };
  karten.push(neueKarte);
  return neueKarte;
}

function listeAnzeigen() {
  const liste = document.querySelector("#karten-liste");
  liste.innerHTML = "";

  for (const karte of karten) {
    const eintrag = document.createElement("li");
    eintrag.textContent = karte.frage;
    liste.appendChild(eintrag);
  }
}

function speichern() {
  await window.__TAURI__.core.invoke("karten_speichern", { karten: karten });
  console.log("gespeichert");
}

// Die Anzeige zu Beginn
listeAnzeigen();

speichern();

`SyntaxError: await is only valid in async functions …` mit Zeilennummer. Übersetzt: *`await` ist nur in `async`-Funktionen erlaubt.* Die Meldung sagt genau, was fehlt. Sie muss nur gelesen werden.

Beachten Sie: Bei einem `SyntaxError` läuft **gar nichts**, auch nicht `listeAnzeigen()`. Der Fehler steckt in der Schreibweise, deshalb wird die ganze Datei nicht ausgeführt. Setzen Sie `async` vor `function` und führen Sie die Zelle erneut aus.

### 🐞 Absichtlicher Fehler: der falsche Name

🤔 Lesen Sie den Namen im `invoke` und den Namen in `main.rs` **laut** nebeneinander.

In [ ]:
// app.js
const karten = [
  { frage: "Was ist HTML?", antwort: "Die Struktur einer Seite" },
  { frage: "Was ist CSS?", antwort: "Das Aussehen einer Seite" },
  { frage: "Was ist eine id?", antwort: "Der Name eines Elements" }
];

let aktuelleKarte = 0;

function karteAnlegen(frage, antwort) {
  const neueKarte = { frage: frage, antwort: antwort };
  karten.push(neueKarte);
  return neueKarte;
}

function listeAnzeigen() {
  const liste = document.querySelector("#karten-liste");
  liste.innerHTML = "";

  for (const karte of karten) {
    const eintrag = document.createElement("li");
    eintrag.textContent = karte.frage;
    liste.appendChild(eintrag);
  }
}

async function speichern() {
  await window.__TAURI__.core.invoke("karten_Speichern", { karten: karten });
  console.log("gespeichert");
}

// Die Anzeige zu Beginn
listeAnzeigen();

speichern();

`Command karten_Speichern not found`: Der Programmkern kennt keine Funktion mit diesem Namen. Groß- und Kleinschreibung zählt, genau wie bei den `id`s in K4. Und `gespeichert` erscheint nicht, weil der Aufruf vorher abgebrochen ist.

Reparieren Sie den Namen.

---
## 5 Laden beim Start

Ab jetzt ist die **Datei** der Datenbestand. Deshalb stehen keine Karten mehr im Programmtext: Die Liste startet **leer** und wird beim Start aus der Datei gefüllt.

🖱️ Führen Sie die Zelle aus. Legen Sie noch **keine** Karte an.

In [ ]:
// app.js
// ★ NEU: Die Karten kommen ab jetzt aus der Datei. Die Liste startet leer.
const karten = [];

let aktuelleKarte = 0;

function karteAnlegen(frage, antwort) {
  const neueKarte = { frage: frage, antwort: antwort };
  karten.push(neueKarte);
  return neueKarte;
}

function listeAnzeigen() {
  const liste = document.querySelector("#karten-liste");
  liste.innerHTML = "";

  for (const karte of karten) {
    const eintrag = document.createElement("li");
    eintrag.textContent = karte.frage;
    liste.appendChild(eintrag);
  }
}

async function speichern() {
  await window.__TAURI__.core.invoke("karten_speichern", { karten: karten });
  console.log("gespeichert");
}

// ★ NEU
async function laden() {
  const geladen = await window.__TAURI__.core.invoke("karten_laden");
  for (const karte of geladen) {
    karten.push(karte);
  }
  listeAnzeigen();
}

// Knopf „Karte anlegen“
const knopfNeu = document.querySelector("#knopf-neu");

knopfNeu.addEventListener("click", function () {
  const frage = document.querySelector("#frage-feld").value;
  const antwort = document.querySelector("#antwort-feld").value;

  if (frage === "" || antwort === "") {
    console.log("Bitte Frage und Antwort ausfüllen");
    return;
  }

  karteAnlegen(frage, antwort);
  listeAnzeigen();

  document.querySelector("#frage-feld").value = "";
  document.querySelector("#antwort-feld").value = "";
});

// Knopf „Umdrehen“
const knopfUmdrehen = document.querySelector("#knopf-umdrehen");

knopfUmdrehen.addEventListener("click", function () {
  const karte = karten[aktuelleKarte];
  if (!karte) {
    console.log("Noch keine Karte da");
    return;
  }

  const frageElement = document.querySelector("#karte-frage");
  const antwortElement = document.querySelector("#karte-antwort");

  if (frageElement.textContent === karte.frage) {
    antwortElement.textContent = karte.antwort;
  } else {
    frageElement.textContent = karte.frage;
  }
});

// Knopf „Nächste“
const knopfWeiter = document.querySelector("#knopf-weiter");

knopfWeiter.addEventListener("click", function () {
  if (karten.length === 0) {
    console.log("Noch keine Karte da");
    return;
  }

  aktuelleKarte = (aktuelleKarte + 1) % karten.length;

  document.querySelector("#karte-frage").textContent = karten[aktuelleKarte].frage;
  document.querySelector("#karte-antwort").textContent = "";
});

// ★ NEU: Beim Start: Karten aus der Datei laden
laden();

Die drei Karten erscheinen, obwohl `karten` leer startet. Sie kommen aus der Datei, die Sie in Abschnitt 4 geschrieben haben. Der Bereich 📄 meldet `karten_laden: 3 Karten gelesen`.

> Steht bei Ihnen „Noch keine Karte“ und die Liste ist leer? Dann gibt es die Datei noch nicht. Führen Sie die Zelle aus Abschnitt 4 einmal aus.

Zeile für Zeile:

| Zeile | Was sie tut |
|---|---|
| `const geladen = await …invoke("karten_laden")` | den Programmkern fragen und die Antwort **abwarten** |
| `for (const karte of geladen)` | jede geladene Karte durchgehen |
| `karten.push(karte);` | sie in die Liste der Anwendung übernehmen |
| `listeAnzeigen();` | die Anzeige neu aufbauen, **sonst ist alles da und unsichtbar** |

> **`listeAnzeigen()` nicht vergessen.** Nach dem Laden sind die Daten da, aber der Bildschirm zeigt noch die leere Liste. Das ist derselbe Fehler wie in K4, jetzt mit einer anderen Ursache.

Der Aufruf `laden();` ganz unten steht **ohne** `await`. Das ist Absicht: Der Start soll nicht warten, das Laden passiert nebenbei.

### 🐞 Warum startet `karten` jetzt leer?

🤔 In dieser Fassung stehen die drei Beispielkarten noch im Programmtext, so wie in K4. Was zeigt die Liste?

In [ ]:
// app.js
const karten = [
  { frage: "Was ist HTML?", antwort: "Die Struktur einer Seite" },
  { frage: "Was ist CSS?", antwort: "Das Aussehen einer Seite" },
  { frage: "Was ist eine id?", antwort: "Der Name eines Elements" }
];

let aktuelleKarte = 0;

function karteAnlegen(frage, antwort) {
  const neueKarte = { frage: frage, antwort: antwort };
  karten.push(neueKarte);
  return neueKarte;
}

function listeAnzeigen() {
  const liste = document.querySelector("#karten-liste");
  liste.innerHTML = "";

  for (const karte of karten) {
    const eintrag = document.createElement("li");
    eintrag.textContent = karte.frage;
    liste.appendChild(eintrag);
  }
}

async function speichern() {
  await window.__TAURI__.core.invoke("karten_speichern", { karten: karten });
  console.log("gespeichert");
}

async function laden() {
  const geladen = await window.__TAURI__.core.invoke("karten_laden");
  for (const karte of geladen) {
    karten.push(karte);
  }
  listeAnzeigen();
}

// Knopf „Karte anlegen“
const knopfNeu = document.querySelector("#knopf-neu");

knopfNeu.addEventListener("click", function () {
  const frage = document.querySelector("#frage-feld").value;
  const antwort = document.querySelector("#antwort-feld").value;

  if (frage === "" || antwort === "") {
    console.log("Bitte Frage und Antwort ausfüllen");
    return;
  }

  karteAnlegen(frage, antwort);
  listeAnzeigen();

  document.querySelector("#frage-feld").value = "";
  document.querySelector("#antwort-feld").value = "";
});

// Knopf „Umdrehen“
const knopfUmdrehen = document.querySelector("#knopf-umdrehen");

knopfUmdrehen.addEventListener("click", function () {
  const karte = karten[aktuelleKarte];
  if (!karte) {
    console.log("Noch keine Karte da");
    return;
  }

  const frageElement = document.querySelector("#karte-frage");
  const antwortElement = document.querySelector("#karte-antwort");

  if (frageElement.textContent === karte.frage) {
    antwortElement.textContent = karte.antwort;
  } else {
    frageElement.textContent = karte.frage;
  }
});

// Knopf „Nächste“
const knopfWeiter = document.querySelector("#knopf-weiter");

knopfWeiter.addEventListener("click", function () {
  if (karten.length === 0) {
    console.log("Noch keine Karte da");
    return;
  }

  aktuelleKarte = (aktuelleKarte + 1) % karten.length;

  document.querySelector("#karte-frage").textContent = karten[aktuelleKarte].frage;
  document.querySelector("#karte-antwort").textContent = "";
});

// Beim Start: Karten aus der Datei laden
laden();

**Sechs** Einträge, jede Karte doppelt: drei aus dem Programmtext, drei aus der Datei. Und es wird schlimmer: Sobald gespeichert wird, stehen alle sechs in der Datei, und beim nächsten Start sind es neun.

Deshalb gilt ab K5: **Der Programmtext enthält keine Karten mehr.** `const karten = [];` und alles Weitere kommt aus der Datei.

---
## 6 Speichern an den richtigen Stellen

Laden funktioniert. 🖱️ Probieren Sie jetzt: Legen Sie eine eigene Karte an und klicken Sie auf **⟳ Neu starten**.

In [ ]:
// app.js
// Die Karten kommen ab jetzt aus der Datei. Die Liste startet leer.
const karten = [];

let aktuelleKarte = 0;

function karteAnlegen(frage, antwort) {
  const neueKarte = { frage: frage, antwort: antwort };
  karten.push(neueKarte);
  return neueKarte;
}

function listeAnzeigen() {
  const liste = document.querySelector("#karten-liste");
  liste.innerHTML = "";

  for (const karte of karten) {
    const eintrag = document.createElement("li");
    eintrag.textContent = karte.frage;
    liste.appendChild(eintrag);
  }
}

async function speichern() {
  await window.__TAURI__.core.invoke("karten_speichern", { karten: karten });
  console.log("gespeichert");
}

async function laden() {
  const geladen = await window.__TAURI__.core.invoke("karten_laden");
  for (const karte of geladen) {
    karten.push(karte);
  }
  listeAnzeigen();
}

// Knopf „Karte anlegen“
const knopfNeu = document.querySelector("#knopf-neu");

knopfNeu.addEventListener("click", function () {
  const frage = document.querySelector("#frage-feld").value;
  const antwort = document.querySelector("#antwort-feld").value;

  if (frage === "" || antwort === "") {
    console.log("Bitte Frage und Antwort ausfüllen");
    return;
  }

  karteAnlegen(frage, antwort);
  listeAnzeigen();

  document.querySelector("#frage-feld").value = "";
  document.querySelector("#antwort-feld").value = "";
});

// Knopf „Umdrehen“
const knopfUmdrehen = document.querySelector("#knopf-umdrehen");

knopfUmdrehen.addEventListener("click", function () {
  const karte = karten[aktuelleKarte];
  if (!karte) {
    console.log("Noch keine Karte da");
    return;
  }

  const frageElement = document.querySelector("#karte-frage");
  const antwortElement = document.querySelector("#karte-antwort");

  if (frageElement.textContent === karte.frage) {
    antwortElement.textContent = karte.antwort;
  } else {
    frageElement.textContent = karte.frage;
  }
});

// Knopf „Nächste“
const knopfWeiter = document.querySelector("#knopf-weiter");

knopfWeiter.addEventListener("click", function () {
  if (karten.length === 0) {
    console.log("Noch keine Karte da");
    return;
  }

  aktuelleKarte = (aktuelleKarte + 1) % karten.length;

  document.querySelector("#karte-frage").textContent = karten[aktuelleKarte].frage;
  document.querySelector("#karte-antwort").textContent = "";
});

// Beim Start: Karten aus der Datei laden
laden();

Ihre Karte ist wieder weg. Geladen wird, aber gespeichert wird nie. Schauen Sie auf 📄: Ihre Karte steht nicht in der Datei.

### 🤔 Die Frage: Wo muss `speichern()` aufgerufen werden?

Diese Suche ist die eigentliche Übung, nicht der Code. Die Antwort sind die Stellen, an denen sich **die Daten ändern**:

| Stelle | Nach was | Gibt es das schon? |
|---|---|---|
| nach dem **Anlegen** einer Karte | eine Karte kam dazu | ja, Knopf „Karte anlegen“ |
| nach dem **Löschen** einer Karte | eine Karte ist weg | kommt in K6 |
| nach dem **Ändern** einer Karte | der Inhalt ist anders | kommt in K6 |

„Umdrehen“ und „Nächste“ ändern nur, **was angezeigt wird**, nicht die Daten. Dort wird nicht gespeichert.

### ✏️ Ihre Aufgabe 6

Bauen Sie in der Zelle oben das Speichern in den Knopf „Karte anlegen“ ein:

1. Setzen Sie hinter `listeAnzeigen();` die Zeile `await speichern();`
2. Führen Sie die Zelle aus. Was meldet die Konsole? Denken Sie an den Merksatz.
3. Reparieren Sie, legen Sie eine Karte an, klicken Sie **⟳ Neu starten**. Bleibt die Karte?

<details>
<summary><b>💡 Lösung 6</b> (zum Aufklappen anklicken)</summary>

Die Funktion im Klick braucht jetzt auch ein `async`, denn in ihr steht ein `await`:

```js
knopfNeu.addEventListener("click", async function () {
  // … Werte holen, prüfen …

  karteAnlegen(frage, antwort);
  listeAnzeigen();
  await speichern();

  document.querySelector("#frage-feld").value = "";
  document.querySelector("#antwort-feld").value = "";
});
```

Ohne `async` meldet die Konsole denselben `SyntaxError` wie in Abschnitt 4.

> **Speichern an nur einer Stelle reicht nicht.** Sobald es in K6 einen Löschknopf gibt, muss auch dort gespeichert werden. Sonst sind gelöschte Karten beim nächsten Start wieder da. Der Fehler ist beim Ausprobieren harmlos und fällt erst später auf.

</details>

### So sieht `app.js` am Ende von K5 aus

🖱️ Die Probe, auf die alles hinausläuft: Legen Sie zwei Karten an, klicken Sie **⟳ Neu starten**, und Ihre Karten sind noch da. Führen Sie die Zelle sogar erneut aus: Sie bleiben.

In [ ]:
// app.js
// Die Karten kommen ab jetzt aus der Datei. Die Liste startet leer.
const karten = [];

let aktuelleKarte = 0;

function karteAnlegen(frage, antwort) {
  const neueKarte = { frage: frage, antwort: antwort };
  karten.push(neueKarte);
  return neueKarte;
}

function listeAnzeigen() {
  const liste = document.querySelector("#karten-liste");
  liste.innerHTML = "";

  for (const karte of karten) {
    const eintrag = document.createElement("li");
    eintrag.textContent = karte.frage;
    liste.appendChild(eintrag);
  }
}

async function speichern() {
  await window.__TAURI__.core.invoke("karten_speichern", { karten: karten });
  console.log("gespeichert");
}

async function laden() {
  const geladen = await window.__TAURI__.core.invoke("karten_laden");
  for (const karte of geladen) {
    karten.push(karte);
  }
  listeAnzeigen();
}

// Knopf „Karte anlegen“
const knopfNeu = document.querySelector("#knopf-neu");

knopfNeu.addEventListener("click", async function () {
  const frage = document.querySelector("#frage-feld").value;
  const antwort = document.querySelector("#antwort-feld").value;

  if (frage === "" || antwort === "") {
    console.log("Bitte Frage und Antwort ausfüllen");
    return;
  }

  karteAnlegen(frage, antwort);
  listeAnzeigen();
  await speichern();

  document.querySelector("#frage-feld").value = "";
  document.querySelector("#antwort-feld").value = "";
});

// Knopf „Umdrehen“
const knopfUmdrehen = document.querySelector("#knopf-umdrehen");

knopfUmdrehen.addEventListener("click", function () {
  const karte = karten[aktuelleKarte];
  if (!karte) {
    console.log("Noch keine Karte da");
    return;
  }

  const frageElement = document.querySelector("#karte-frage");
  const antwortElement = document.querySelector("#karte-antwort");

  if (frageElement.textContent === karte.frage) {
    antwortElement.textContent = karte.antwort;
  } else {
    frageElement.textContent = karte.frage;
  }
});

// Knopf „Nächste“
const knopfWeiter = document.querySelector("#knopf-weiter");

knopfWeiter.addEventListener("click", function () {
  if (karten.length === 0) {
    console.log("Noch keine Karte da");
    return;
  }

  aktuelleKarte = (aktuelleKarte + 1) % karten.length;

  document.querySelector("#karte-frage").textContent = karten[aktuelleKarte].frage;
  document.querySelector("#karte-antwort").textContent = "";
});

// Beim Start: Karten aus der Datei laden
laden();

---
## 7 🐞 Absichtlicher Fehler: das fehlende `await`

In `laden()` fehlt das `await`. Zum Vergleichen gibt die Funktion aus, was sie vom Programmkern bekommt. 🤔 **Vorhersage:** Was steht in der Konsole?

In [ ]:
// app.js
// Die Karten kommen ab jetzt aus der Datei. Die Liste startet leer.
const karten = [];

let aktuelleKarte = 0;

function karteAnlegen(frage, antwort) {
  const neueKarte = { frage: frage, antwort: antwort };
  karten.push(neueKarte);
  return neueKarte;
}

function listeAnzeigen() {
  const liste = document.querySelector("#karten-liste");
  liste.innerHTML = "";

  for (const karte of karten) {
    const eintrag = document.createElement("li");
    eintrag.textContent = karte.frage;
    liste.appendChild(eintrag);
  }
}

async function laden() {
  const geladen = window.__TAURI__.core.invoke("karten_laden");
  console.log(geladen);
  for (const karte of geladen) {
    karten.push(karte);
  }
  listeAnzeigen();
}

// Beim Start: Karten aus der Datei laden
laden();

In der Konsole steht **`Promise { <pending> }`** statt der Liste, und danach `TypeError: geladen is not iterable`.

> „Das Programm war zu schnell und hat nicht gewartet. Der Programmkern arbeitet im Hintergrund, aber das Fenster wollte sofort weiter und hatte die Antwort noch nicht.“

Ein `Promise` ist ein **Versprechen**: „Die Antwort kommt noch.“ Ohne `await` bekommen Sie das Versprechen, nicht die Antwort. Mit `await` warten Sie, bis das Versprechen eingelöst ist.

Setzen Sie das `await` wieder vor `window.__TAURI__…` und führen Sie die Zelle erneut aus. Jetzt steht in der Konsole die Liste.

---
## 8 Fehlerjagd

In jeder Zelle steckt **genau ein** Fehler. Darüber steht, was passieren **soll**. Schauen Sie auf Konsole **und** 📄.

**a)** 🖱️ Eine neue Karte soll angelegt und gespeichert werden.

In [ ]:
// app.js
// Die Karten kommen ab jetzt aus der Datei. Die Liste startet leer.
const karten = [];

let aktuelleKarte = 0;

function karteAnlegen(frage, antwort) {
  const neueKarte = { frage: frage, antwort: antwort };
  karten.push(neueKarte);
  return neueKarte;
}

function listeAnzeigen() {
  const liste = document.querySelector("#karten-liste");
  liste.innerHTML = "";

  for (const karte of karten) {
    const eintrag = document.createElement("li");
    eintrag.textContent = karte.frage;
    liste.appendChild(eintrag);
  }
}

async function speichern() {
  await window.__TAURI__.core.invoke("karten_speichern", { karten: karten });
  console.log("gespeichert");
}

async function laden() {
  const geladen = await window.__TAURI__.core.invoke("karten_laden");
  for (const karte of geladen) {
    karten.push(karte);
  }
  listeAnzeigen();
}

// Knopf „Karte anlegen“
const knopfNeu = document.querySelector("#knopf-neu");

knopfNeu.addEventListener("click", function () {
  const frage = document.querySelector("#frage-feld").value;
  const antwort = document.querySelector("#antwort-feld").value;

  if (frage === "" || antwort === "") {
    console.log("Bitte Frage und Antwort ausfüllen");
    return;
  }

  karteAnlegen(frage, antwort);
  listeAnzeigen();
  await speichern();

  document.querySelector("#frage-feld").value = "";
  document.querySelector("#antwort-feld").value = "";
});

// Beim Start: Karten aus der Datei laden
laden();

**b)** Beim Start sollen die Karten aus der Datei erscheinen.

In [ ]:
// app.js
// Die Karten kommen ab jetzt aus der Datei. Die Liste startet leer.
const karten = [];

let aktuelleKarte = 0;

function karteAnlegen(frage, antwort) {
  const neueKarte = { frage: frage, antwort: antwort };
  karten.push(neueKarte);
  return neueKarte;
}

function listeAnzeigen() {
  const liste = document.querySelector("#karten-liste");
  liste.innerHTML = "";

  for (const karte of karten) {
    const eintrag = document.createElement("li");
    eintrag.textContent = karte.frage;
    liste.appendChild(eintrag);
  }
}

async function laden() {
  const geladen = await window.__TAURI__.core.invoke("karten_lade");
  for (const karte of geladen) {
    karten.push(karte);
  }
  listeAnzeigen();
}

// Beim Start: Karten aus der Datei laden
laden();

**c)** Beim Start sollen die Karten aus der Datei erscheinen.

In [ ]:
// app.js
// Die Karten kommen ab jetzt aus der Datei. Die Liste startet leer.
const karten = [];

let aktuelleKarte = 0;

function karteAnlegen(frage, antwort) {
  const neueKarte = { frage: frage, antwort: antwort };
  karten.push(neueKarte);
  return neueKarte;
}

function listeAnzeigen() {
  const liste = document.querySelector("#karten-liste");
  liste.innerHTML = "";

  for (const karte of karten) {
    const eintrag = document.createElement("li");
    eintrag.textContent = karte.frage;
    liste.appendChild(eintrag);
  }
}

async function laden() {
  const geladen = await window.__TAURI__.core.invoke("karten_laden");
  for (const karte of geladen) {
    karten.push(karte);
  }
  console.log("Karten im Datenbestand: " + karten.length);
}

// Beim Start: Karten aus der Datei laden
laden();

**d)** Beim Start sollen die drei Beispielkarten gespeichert werden.

In [ ]:
// app.js
const karten = [
  { frage: "Was ist HTML?", antwort: "Die Struktur einer Seite" },
  { frage: "Was ist CSS?", antwort: "Das Aussehen einer Seite" },
  { frage: "Was ist eine id?", antwort: "Der Name eines Elements" }
];

let aktuelleKarte = 0;

function karteAnlegen(frage, antwort) {
  const neueKarte = { frage: frage, antwort: antwort };
  karten.push(neueKarte);
  return neueKarte;
}

function listeAnzeigen() {
  const liste = document.querySelector("#karten-liste");
  liste.innerHTML = "";

  for (const karte of karten) {
    const eintrag = document.createElement("li");
    eintrag.textContent = karte.frage;
    liste.appendChild(eintrag);
  }
}

async function speichern() {
  await window.__TAURI__.core.invoke("karten_speichern", { liste: karten });
  console.log("gespeichert");
}

// Die Anzeige zu Beginn
listeAnzeigen();

speichern();

**e)** 🖱️ Nach dem Anlegen einer Karte soll die **neue** Karte in der Datei stehen. Legen Sie eine an und schauen Sie in 📄.

In [ ]:
// app.js
// Die Karten kommen ab jetzt aus der Datei. Die Liste startet leer.
const karten = [];

let aktuelleKarte = 0;

function karteAnlegen(frage, antwort) {
  const neueKarte = { frage: frage, antwort: antwort };
  karten.push(neueKarte);
  return neueKarte;
}

function listeAnzeigen() {
  const liste = document.querySelector("#karten-liste");
  liste.innerHTML = "";

  for (const karte of karten) {
    const eintrag = document.createElement("li");
    eintrag.textContent = karte.frage;
    liste.appendChild(eintrag);
  }
}

async function speichern() {
  await window.__TAURI__.core.invoke("karten_speichern", { karten: karten });
  console.log("gespeichert");
}

async function laden() {
  const geladen = await window.__TAURI__.core.invoke("karten_laden");
  for (const karte of geladen) {
    karten.push(karte);
  }
  listeAnzeigen();
}

// Knopf „Karte anlegen“
const knopfNeu = document.querySelector("#knopf-neu");

knopfNeu.addEventListener("click", async function () {
  const frage = document.querySelector("#frage-feld").value;
  const antwort = document.querySelector("#antwort-feld").value;

  if (frage === "" || antwort === "") {
    console.log("Bitte Frage und Antwort ausfüllen");
    return;
  }

  await speichern();
  karteAnlegen(frage, antwort);
  listeAnzeigen();

  document.querySelector("#frage-feld").value = "";
  document.querySelector("#antwort-feld").value = "";
});

// Beim Start: Karten aus der Datei laden
laden();

<details>
<summary><b>💡 Lösungen zur Fehlerjagd</b> (zum Aufklappen anklicken)</summary>

| | Was Sie sehen | Ursache | Reparatur |
|---|---|---|---|
| a) | `SyntaxError: await is only valid in async functions …`, das ganze Fenster tut nichts | `await speichern()` in einer Funktion ohne `async` | `async function () {` |
| b) | `Command karten_lade not found` | Name passt nicht zu `main.rs` | `"karten_laden"` |
| c) | Konsole meldet 3 Karten, die Liste im Fenster bleibt leer | `listeAnzeigen()` fehlt nach dem Laden | `listeAnzeigen();` am Ende von `laden()` |
| d) | ``invalid args `karten` for command `karten_speichern` …`` | links in `{ … }` muss der Name aus `main.rs` stehen | `{ karten: karten }` |
| e) | die neue Karte fehlt in der Datei, sie kommt erst beim nächsten Anlegen dazu | erst gespeichert, dann angelegt | `await speichern();` nach `karteAnlegen` |

Bei c) und e) gibt es **keine** Fehlermeldung. Bei e) merken Sie den Fehler erst nach einem Neustart: Die zuletzt angelegte Karte ist weg. Wer auf 📄 schaut, sieht es sofort.

Wenn die Datei durch die Fehlerjagd durcheinandergeraten ist: **🗑 Datei löschen** und die Zelle aus Abschnitt 4 einmal ausführen.

</details>

---
## 9 Wo liegt die Datei? Und eine Werkstatt für Schnellere

### In der echten App

Der Programmkern schreibt in den **Anwendungsdatenordner**, nicht in Ihren Projektordner. Unter Linux liegt er in Ihrem Home-Verzeichnis unter `.local/share/` und heißt wie die Kennung der App:

```nu
ls ~/.local/share/de.metarow.Lernkarten
```

Und so sehen Sie den Inhalt. Nushell zeigt eine JSON-Datei sogar gleich als Tabelle an:

```nu
open ~/.local/share/de.metarow.Lernkarten/lernkarten.json
```

> **Wer im Projektordner sucht, findet nichts.** Die App darf in „ihren“ Ordner schreiben, nicht irgendwohin. Das ist dieselbe Idee wie der Schlüssel für die Schränke: Sie hat den Schlüssel für **ihren** Schrank.

Hier im Notebook ist der Bereich 📄 Ihr Blick in diese Datei.

### Werkstatt für Schnellere

Kein neuer Stoff, nur Verbesserungen mit dem, was Sie schon können. Arbeiten Sie in der Zelle unten.

1. **Gleich etwas zeigen:** Nach dem Laden soll sofort die Frage der ersten Karte in `#karte-frage` stehen, falls es Karten gibt.
2. **Willkommenskarten:** Ist die Datei beim Start leer (frisch installiert), legt die App drei Startkarten an und speichert sie. Probieren Sie es mit **🗑 Datei löschen** und **⟳ Neu starten**.

In [ ]:
// app.js
// Die Karten kommen ab jetzt aus der Datei. Die Liste startet leer.
const karten = [];

let aktuelleKarte = 0;

function karteAnlegen(frage, antwort) {
  const neueKarte = { frage: frage, antwort: antwort };
  karten.push(neueKarte);
  return neueKarte;
}

function listeAnzeigen() {
  const liste = document.querySelector("#karten-liste");
  liste.innerHTML = "";

  for (const karte of karten) {
    const eintrag = document.createElement("li");
    eintrag.textContent = karte.frage;
    liste.appendChild(eintrag);
  }
}

async function speichern() {
  await window.__TAURI__.core.invoke("karten_speichern", { karten: karten });
  console.log("gespeichert");
}

async function laden() {
  const geladen = await window.__TAURI__.core.invoke("karten_laden");
  for (const karte of geladen) {
    karten.push(karte);
  }
  listeAnzeigen();
}

// Knopf „Karte anlegen“
const knopfNeu = document.querySelector("#knopf-neu");

knopfNeu.addEventListener("click", async function () {
  const frage = document.querySelector("#frage-feld").value;
  const antwort = document.querySelector("#antwort-feld").value;

  if (frage === "" || antwort === "") {
    console.log("Bitte Frage und Antwort ausfüllen");
    return;
  }

  karteAnlegen(frage, antwort);
  listeAnzeigen();
  await speichern();

  document.querySelector("#frage-feld").value = "";
  document.querySelector("#antwort-feld").value = "";
});

// Knopf „Umdrehen“
const knopfUmdrehen = document.querySelector("#knopf-umdrehen");

knopfUmdrehen.addEventListener("click", function () {
  const karte = karten[aktuelleKarte];
  if (!karte) {
    console.log("Noch keine Karte da");
    return;
  }

  const frageElement = document.querySelector("#karte-frage");
  const antwortElement = document.querySelector("#karte-antwort");

  if (frageElement.textContent === karte.frage) {
    antwortElement.textContent = karte.antwort;
  } else {
    frageElement.textContent = karte.frage;
  }
});

// Knopf „Nächste“
const knopfWeiter = document.querySelector("#knopf-weiter");

knopfWeiter.addEventListener("click", function () {
  if (karten.length === 0) {
    console.log("Noch keine Karte da");
    return;
  }

  aktuelleKarte = (aktuelleKarte + 1) % karten.length;

  document.querySelector("#karte-frage").textContent = karten[aktuelleKarte].frage;
  document.querySelector("#karte-antwort").textContent = "";
});

// Beim Start: Karten aus der Datei laden
laden();

<details>
<summary><b>💡 Lösungsideen</b> (zum Aufklappen anklicken)</summary>

Beide Ideen gehören in `laden()`, hinter die Schleife:

```js
async function laden() {
  const geladen = await window.__TAURI__.core.invoke("karten_laden");
  for (const karte of geladen) {
    karten.push(karte);
  }

  // 2. Willkommenskarten
  if (karten.length === 0) {
    karteAnlegen("Was ist HTML?", "Die Struktur einer Seite");
    karteAnlegen("Was ist CSS?", "Das Aussehen einer Seite");
    karteAnlegen("Was ist JavaScript?", "Das Verhalten einer Seite");
    await speichern();
  }

  listeAnzeigen();

  // 1. Gleich etwas zeigen
  if (karten.length > 0) {
    document.querySelector("#karte-frage").textContent = karten[0].frage;
  }
}
```

`await speichern()` geht hier ohne Weiteres, denn `laden()` ist schon `async`.

</details>

---
## 10 Diagnosetabelle, Checkliste und Ausblick

### Die Diagnosetabelle, erweitert um K5

| Was zu sehen ist | Was das meistens heißt |
|---|---|
| `SyntaxError: await is only valid in async functions` | `async` fehlt vor der Funktion, in der `await` steht |
| `Promise { <pending> }` statt einer Liste | `await` vergessen |
| `… is not iterable` direkt nach einem `invoke` | `await` vergessen |
| `Command … not found` | Name im `invoke` passt nicht zu `main.rs` |
| `invalid args … missing required key …` | Name links in `{ … }` passt nicht zu `main.rs` |
| Daten geladen (Konsole), Liste leer | `listeAnzeigen()` fehlt nach dem Laden |
| Karten nach dem Neustart weg | an dieser Stelle wird nicht gespeichert |
| Karten stehen nach jedem Start doppelt | Beispielkarten stehen noch im Programmtext |
| Datei im Projektordner nicht gefunden | sie liegt im Anwendungsdatenordner `~/.local/share/de.metarow.Lernkarten` |

### ✅ Checkliste

Haken Sie ab (Doppelklick auf diese Zelle, dann `[ ]` zu `[x]` ändern):

- [ ] Ich kann erklären, warum eine Webseite keine Dateien schreiben darf und eine installierte App schon
- [ ] Ich kann die beiden Funktionen aus `main.rs` in eigenen Worten beschreiben
- [ ] Ich habe `speichern()` geschrieben und die Datei `lernkarten.json` angesehen
- [ ] Ich habe `laden()` geschrieben und beim Start aufgerufen
- [ ] Ich weiß, warum `karten` jetzt leer startet
- [ ] Ich habe alle Stellen gefunden, an denen gespeichert werden muss
- [ ] Meine Karten überleben den Neustart
- [ ] Ich habe den Merksatz parat: **Wo `await` steht, muss `async` davorstehen.**

### Wie geht es weiter?

Ihre App kann jetzt alles, was sie können muss. In **K6 „Feinschliff“** machen Sie sie zu Ihrer eigenen: eigene Gestaltung und eine kleine Verbesserung, zum Beispiel ein Löschknopf für einzelne Karten. Denken Sie dann an diese Seite: **Nach dem Löschen muss gespeichert werden.**

**Zum Nachschlagen:** K5 Merkblatt · K5 Übungsblatt · Anhang Befehlskarte